In [103]:
! bash ./src/setup-utils.sh
import sys  
sys.path.insert(0, './src')

<3>WSL (9 - Relay) ERROR: CreateProcessCommon:800: execvpe(/bin/bash) failed: No such file or directory


In [104]:
%load_ext autoreload
%autoreload 2
%cd ./

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
c:\Users\kayri\OneDrive - IIT BHU\Documents\Brand-Sentiment-Analysis-main


In [105]:
import pandas as pd
import brands,utils
import Article_Binary_Classifier_Inference
import Tweet_Binary_Classifier_Inference
import headline_generation
import sentiment_inference
import time
import pickle

In [106]:
!wget -O models/tweet_vect.pkl "https://drive.google.com/uc?export=download&id=1-BDg4YIu-CNHCwBUMhyXkEsun3M0Y3wQ"
!wget -O models/tweet_classf.pkl "https://drive.google.com/uc?export=download&id=1YsQO7LP5ezvDeh9MUlW6HRwKvWkXDYF6"
!wget -O models/article_vect.pkl "https://drive.google.com/uc?export=download&id=174SJ90A4mBpPa1gQIylS63k4e5kE41Wt"
!wget -O models/article_classf.pkl "https://drive.google.com/uc?export=download&id=1LqlpZNgho3unOhAfjdai6GQInBkjX7VJ"
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1-3OLtOIbJ6rhYpmbGdpaUuvNHyttqPWq' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1-3OLtOIbJ6rhYpmbGdpaUuvNHyttqPWq" -O ./models/T5-headline.pth && rm -rf /tmp/cookies.txt
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1uRzeh66xQnH-dIkzISRL1c1spQnTG-uA' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1uRzeh66xQnH-dIkzISRL1c1spQnTG-uA" -O ./models/mbert-for-sentiment.pth && rm -rf /tmp/cookies.txt
!wget -O models/regression.pkl "https://drive.google.com/uc?export=download&id=1Hdck45_FQpt7D7lrEs3IvQ-lAqKeMDIX"
!wget -O models/tfidf.pkl "https://drive.google.com/uc?export=download&id=1jsN5kaj3tcCsxNEiF8DGLkmjwwFdG9IJ"

'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [107]:
df = pd.read_excel('./data/evaluation_data.xlsx')
df.head()

,Text_ID,Text
0,article_0001,Digitisation is one of the key buzzwords in th...
1,article_0002,Increase in tolerance limit up to 120 per cent...
2,article_0003,Home > News > World Sports News\n\nJonas Lossl...
3,article_0004,"Source: Agfax.com\n\nBy Keith Brown, DTN Contr..."
4,article_0005,"United Nations, Feb 4: The Serum Institute of ..."


In [108]:
df_articles = df[df['Text_ID'].str.contains('article')].copy().reset_index(drop=True)
df_tweets = df[df['Text_ID'].str.contains('tweet')].copy().reset_index(drop=True)

In [109]:
df_articles.head()

,Text_ID,Text
0,article_0001,Digitisation is one of the key buzzwords in th...
1,article_0002,Increase in tolerance limit up to 120 per cent...
2,article_0003,Home > News > World Sports News\n\nJonas Lossl...
3,article_0004,"Source: Agfax.com\n\nBy Keith Brown, DTN Contr..."
4,article_0005,"United Nations, Feb 4: The Serum Institute of ..."


In [110]:
df_tweets.head()

,Text_ID,Text
0,tweet_0001,You'll 💜 my #PitchWars book if you like: 🦋 hat...
1,tweet_0002,RT @SkySportsNews: 🚨 Breaking: #WBA have reach...
2,tweet_0003,RT @stealyoman_cuso: really says a lot about s...
3,tweet_0004,RT @PGtzsche1: HPV vaccines increased serious ...
4,tweet_0005,Ramaphosa says if you are positive you must se...


In [111]:
start = time.time()
df_articles['Text'] = utils.clean_articles(df_articles['Text'].values)
df_tweets['Text'] = utils.clean_tweets(df_tweets['Text'].values, remove_emoji=False)
df_articles['brands'] = brands.get_brands(df_articles['Text'].values)
df_articles['num_brands'] = df_articles['brands'].apply(lambda x: len(x))
df_tweets['brands'] = brands.get_brands(df_tweets['Text'].values)
df_tweets['num_brands'] = df_tweets['brands'].apply(lambda x: len(x))
end = time.time()
PREPROCESS_TIME = end - start
print(PREPROCESS_TIME)

100%|██████████| 4000/4000 [00:07<00:00, 506.28it/s] 

68.99926614761353


In [112]:
start = time.time()
df_tweets = Tweet_Binary_Classifier_Inference.mobile_tech_binary_classifier(df_tweets)
df_articles = Article_Binary_Classifier_Inference.mobile_tech_binary_classifier(df_articles)
end = time.time()
BINARY_CLASSIFICATION_TIME = end-start
print(BINARY_CLASSIFICATION_TIME)

5.444305181503296


In [113]:
df_tweets['Mobile_Tech'].value_counts()

Mobile_Tech
1    4000
Name: count, dtype: int64

In [114]:
df_articles['Mobile_Tech'].value_counts()

Mobile_Tech
1    3875
Name: count, dtype: int64

In [115]:
df_articles_mob = df_articles[df_articles['Mobile_Tech']==1].copy().reset_index(drop=True)
df_tweets_mob = df_tweets[df_tweets['Mobile_Tech']==1].copy().reset_index(drop=True)

In [116]:
len(df_articles_mob), len(df_tweets_mob)

(3875, 4000)

In [117]:
start = time.time()
df_articles_mob['lang'] = utils.detect_lang(df_articles_mob['Text'].values)
df_tweets_mob['lang'] = utils.detect_lang(df_tweets_mob['Text'].values)
end = time.time()
LANGUAGE_DETECTION_TIME = end - start
print(LANGUAGE_DETECTION_TIME)

100%|██████████| 4000/4000 [2:24:50<00:00,  2.17s/it]       


9453.222054243088


In [118]:
df_articles_mob['lang'].value_counts()

lang
en      2059
hi      1677
hing     139
Name: count, dtype: int64

In [119]:
df_tweets_mob['lang'].value_counts()

lang
en      2004
hi      1313
hing     683
Name: count, dtype: int64

In [120]:
start = time.time()
df_articles_mob['Text'] = brands.replace_hin_to_eng(df_articles_mob['Text'].values)
df_tweets_mob['Text'] = brands.replace_hin_to_eng(df_tweets_mob['Text'].values)
df_articles_mob['brands'] = brands.get_brands(df_articles_mob['Text'].values)
df_articles_mob['num_brands'] = df_articles_mob['brands'].apply(lambda x: len(x))
df_tweets_mob['brands'] = brands.get_brands(df_tweets_mob['Text'].values)
df_tweets_mob['num_brands'] = df_tweets_mob['brands'].apply(lambda x: len(x))
end = time.time()
PREPROCESS2_TIME = end - start
print(PREPROCESS2_TIME)

100%|██████████| 4000/4000 [00:03<00:00, 1225.10it/s]

46.599472761154175


In [ ]:
start = time.time()
# Hindi translation using Free Google API
df_articles_mob.loc[df_articles_mob['lang']=='hi','Text'] = utils.translate(df_articles_mob[df_articles_mob['lang']=='hi']['Text'].values)[0]

# Truncating Hinglish Articles for faster translation
df_articles_mob.loc[df_articles_mob['lang']=='hing','Text'] = df_articles_mob[df_articles_mob['lang']=='hing']['Text'].apply(lambda x: x[:3500])

# Hinglish Translation, takes a lot of time due to use of free API
df_articles_mob.loc[df_articles_mob['lang']=='hing','Text'] = utils.translate(df_articles_mob[df_articles_mob['lang']=='hing']['Text'].values, hinglish=True)[0]
#Double translation works better in case of Hinglish sometimes
# Hinglish Translation, takes a lot of time due to use of free API
df_articles_mob.loc[df_articles_mob['lang']=='hing','Text'] = utils.translate(df_articles_mob[df_articles_mob['lang']=='hing']['Text'].values, hinglish=True)[0]

# Hindi translation using Free Google API
df_tweets_mob.loc[df_tweets_mob['lang']=='hi','Text'] = utils.translate(df_tweets_mob[df_tweets_mob['lang']=='hi']['Text'].values)[0]

# Hinglish translation using Free Google API
df_tweets_mob.loc[df_tweets_mob['lang']=='hing','Text'] = utils.translate(df_tweets_mob[df_tweets_mob['lang']=='hing']['Text'].values)[0]
end = time.time()
TRANSLATION_TIME = end-start
print(TRANSLATION_TIME)
# This can be greatly improved by buying a paid API service

 77%|███████▋  | 1297/1677 [2:56:14<32:18,  5.10s/it]      

In [ ]:
df_articles_mob.to_pickle('./df_articles_mob_cache.pkl')
df_tweets_mob.to_pickle('./df_tweets_mob_cache.pkl')

In [ ]:
df_articles_mob = pd.read_pickle('./df_articles_mob_cache.pkl')

In [ ]:
df_tweets_mob = pd.read_pickle('./df_tweets_mob_cache.pkl')

In [ ]:
start = time.time()
headline_generator = headline_generation.headline_gen(device='cuda',path = './models/T5-headline.pth')
df_articles_mob['headlines'] = headline_generator.predict(df_articles_mob['Text'].values)
end = time.time()
HEADLINE_GENERATION_TIME = end-start
print(HEADLINE_GENERATION_TIME)

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 866.89it/s]


9.01891803741455


In [ ]:
df_articles_mob[['Text','headlines']]

,Text,headlines
0,Apple unveiled its latest flagship iPhone feat...,its latest flagship iPhone featuring an upgrad...
1,Samsung announced its new Galaxy S24 lineup wi...,its new Galaxy S24 lineup with advanced Galaxy...


In [ ]:
df_articles['headline'] = ''
df_articles.loc[df_articles['Mobile_Tech']==1,'headline'] = df_articles_mob['headlines'].values

In [ ]:
df_headline_ouput = df_articles[['Text_ID','Mobile_Tech','headline']].copy().reset_index(drop=True)

In [ ]:
df_headline_ouput.rename(columns = {'Mobile_Tech':'Mobile_Tech_Flag_Predicted','headline':'Headline_Generated_Eng_Lang'}, inplace = True)

In [ ]:
df_headline_ouput.to_csv('../headline-output.csv',index=False)

In [ ]:
# PREPROCESS3 for passing data to sentiment pipeline
start = time.time()
article_brand_sent_dicts = []
for row in df_articles_mob[df_articles_mob['num_brands']>0].iterrows():
  text_id = row[1].iloc[0]
  text = row[1].iloc[1]
  brand_dict = utils.segment_by_rule(text)
  brand_dict['Text_ID'] = text_id
  article_brand_sent_dicts.append(brand_dict)

tweet_brand_sent_dicts = []
for row in df_tweets_mob[df_tweets_mob['num_brands']>0].iterrows():
  text_id = row[1].iloc[0]
  text = row[1].iloc[1]
  tweet_dict = dict()
  tweet_dict['Text_ID'] = text_id
  tweet_dict['Text'] = text
  tweet_brand_sent_dicts.append(tweet_dict)
end = time.time()
PREPROCESS3_TIME = end - start
print(PREPROCESS3_TIME)

with open('./article_brand_sent_dicts.pkl', 'wb') as f:
    pickle.dump(article_brand_sent_dicts, f)

with open('./tweet_brand_sent_dicts.pkl', 'wb') as f:
    pickle.dump(tweet_brand_sent_dicts, f)

0.016635656356811523


In [ ]:
with open('./article_brand_sent_dicts.pkl', 'rb') as f:
  article_brand_sent_dicts = pickle.load(f)
with open('./tweet_brand_sent_dicts.pkl', 'rb') as f:
  tweet_brand_sent_dicts = pickle.load(f)

In [ ]:
sentiment = sentiment_inference.SentimentClassifier(bert_path="./models/mbert-for-sentiment.pth")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1239.41it/s]


In [ ]:
start = time.time()
article_sents_out = sentiment.predict(article_brand_sent_dicts,is_tweets=False)
tweets_sents_out = sentiment.predict(tweet_brand_sent_dicts,is_tweets=True)
end = time.time()
SENTIMENT_TIME = end - start
print(SENTIMENT_TIME)

0.5265254974365234


In [ ]:
 df_articles.loc[df_articles['Text_ID']=='article_4008']

,Text_ID,Text,brands,num_brands,Mobile_Tech,headline


In [ ]:
df_articles['sent_dict'] = [dict() for _ in range(len(df_articles))]
df_tweets['sent_dict'] = [dict() for _ in range(len(df_tweets))]

article_sent_map = {d['Text_ID']: {k: v for k, v in d.items() if k != 'Text_ID'} for d in article_sents_out}
df_articles['sent_dict'] = df_articles['Text_ID'].map(lambda tid: article_sent_map.get(tid, {}))

tweet_sent_map = {d['Text_ID']: {k: v for k, v in d.items() if k != 'Text_ID'} for d in tweets_sents_out}
df_tweets['sent_dict'] = df_tweets['Text_ID'].map(lambda tid: tweet_sent_map.get(tid, {}))


In [ ]:
a = 'Text_ID'
b = 'Mobile_Tech_Flag'
c = 'Brands_Entity_Identified'
d = 'Sentiment_Identified'
df_sentiment_output = []
for row in df_tweets.iterrows():
  text_id = row[1]['Text_ID']
  mob = row[1]['Mobile_Tech']
  brandlist = row[1]['brands']
  sent_dict = row[1]['sent_dict']
  if not isinstance(sent_dict, dict):
     sent_dict = {}
  if len(brandlist)==0:
     df_sentiment_output.append({a:text_id,b:mob,c:'',d:''})
  else:
    for brandname in brandlist:
      brandsent = 'Neutral'
      if brandname in sent_dict.keys():
        if sent_dict[brandname] == 0:
          brandsent = 'Negative'
        elif sent_dict[brandname] == 2:
          brandsent = 'Positive'
      df_sentiment_output.append({a:text_id,b:mob,c:brandname,d:brandsent})

for row in df_articles.iterrows():
  text_id = row[1]['Text_ID']
  mob = row[1]['Mobile_Tech']
  brandlist = row[1]['brands']
  sent_dict = row[1]['sent_dict']
  if not isinstance(sent_dict, dict):
     sent_dict = {}
  if len(brandlist)==0:
    df_sentiment_output.append({a:text_id,b:mob,c:'',d:''})
  else:
    for brandname in brandlist:
      brandsent = 'Neutral'
      if brandname in sent_dict.keys():
        if sent_dict[brandname] == 0:
          brandsent = 'Negative'
        elif sent_dict[brandname] == 2:
          brandsent = 'Positive'
      df_sentiment_output.append({a:text_id,b:mob,c:brandname,d:brandsent})


In [ ]:
print(article_brand_sent_dicts)
print(tweet_brand_sent_dicts)

[{'apple': ['Apple unveiled its latest flagship i Phone featuring an upgraded camera system , A17 Bionic chip , and titanium body.']}, {'samsung': ['Samsung announced its new Galaxy S24 lineup with advanced Galaxy AI features and brighter AMOLED displays.']}]
[{'Text': 'Just bought the new Samsung Galaxy! The display quality is incredible. #Samsung #Galaxy'}, {'Text': 'Apple iPhone 15 battery life has been surprisingly good so far. #Apple'}, {'Text': 'Not really impressed with the new Xiaomi phone design, feels cheap. #Xiaomi'}]


In [ ]:
df_sentiment_output

[{'Text_ID': 'tweet_5001',
  'Mobile_Tech_Flag': 1,
  'Brands_Entity_Identified': 'samsung',
  'Sentiment_Identified': 'Neutral'},
 {'Text_ID': 'tweet_5002',
  'Mobile_Tech_Flag': 1,
  'Brands_Entity_Identified': 'apple',
  'Sentiment_Identified': 'Neutral'},
 {'Text_ID': 'tweet_5003',
  'Mobile_Tech_Flag': 1,
  'Brands_Entity_Identified': 'xiaomi',
  'Sentiment_Identified': 'Neutral'},
 {'Text_ID': 'article_4001',
  'Mobile_Tech_Flag': 1,
  'Brands_Entity_Identified': 'apple',
  'Sentiment_Identified': 'Positive'},
 {'Text_ID': 'article_4002',
  'Mobile_Tech_Flag': 1,
  'Brands_Entity_Identified': 'samsung',
  'Sentiment_Identified': 'Positive'}]

In [ ]:
df_sentiment_output = pd.DataFrame(df_sentiment_output)

In [ ]:
df_sentiment_output

,Text_ID,Mobile_Tech_Flag,Brands_Entity_Identified,Sentiment_Identified
0,tweet_5001,1,samsung,Neutral
1,tweet_5002,1,apple,Neutral
2,tweet_5003,1,xiaomi,Neutral
3,article_4001,1,apple,Positive
4,article_4002,1,samsung,Positive


In [ ]:
import os
os.makedirs('./sample_data', exist_ok=True)
df_sentiment_output.to_csv('./sample_data/sentiment-output.csv', index=False)


In [ ]:
RUNTIME = PREPROCESS_TIME \
  + BINARY_CLASSIFICATION_TIME \
  + LANGUAGE_DETECTION_TIME \
  + PREPROCESS2_TIME \
  + TRANSLATION_TIME \
  + HEADLINE_GENERATION_TIME \
  + PREPROCESS3_TIME \
  + SENTIMENT_TIME


In [ ]:
print('Total Runtime:',RUNTIME)

Total Runtime: 10.675924301147461
